In [ ]:
# import json, os, pickle
# import numpy as np
# from tqdm.notebook import tqdm
# import pandas as pd

# import torch
# from torch.utils.data import DataLoader, TensorDataset
# import matplotlib.pyplot as plt
# import seaborn as sns
# import qiskit.circuit.random
# import torch, random
# from torch.utils.data import Dataset, DataLoader, TensorDataset
# from torch.optim.lr_scheduler import ReduceLROnPlateau
# import torch.nn as nn

# import numpy as np
# import json, os, pickle
# from tqdm import tqdm
# import pandas as pd

# import matplotlib.pyplot as plt
# import seaborn as sns

# from qiskit import QuantumCircuit

# import sys
# sys.path.append('../tutorials/')
# from mlp import encode_data, encode_data_v2_ecr

In [1]:
import os
import qiskit
from qiskit.circuit import QuantumCircuit
from qiskit.qpy import load
from typing import List
from tqdm import tqdm

In [2]:
def load_qpy_circuits_from_folder(folder_path: str) -> List[QuantumCircuit]:
    """
    Load all .qpy circuit files from a specified folder.

    Parameters:
        folder_path (str): The path to the folder containing .qpy files.

    Returns:
        List[QuantumCircuit]: A list of QuantumCircuit objects.
    """
    circuits = []

    # Ensure folder exists
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    # Loop through all files in the directory
    for filename in tqdm(os.listdir(folder_path)):
        if filename.endswith(".qpy"):
            qpy_path = os.path.join(folder_path, filename)
            try:
                with open(qpy_path, "rb") as f:
                    # A .qpy file can contain multiple circuits
                    circuits.extend(load(f))
            except:
                # print
                continue
        # break

    return circuits

In [3]:
# folder = "../../../andrew/ExecutionResults/StoredCircuits/"
folder = "/home/alitousi/projects/quantum/ExecutionResults/StoredCircuits/"
loaded_circuits = load_qpy_circuits_from_folder(folder)


100%|█████████████████████████████████████████████████████████████████████████████████████████████| 7004/7004 [00:13<00:00, 509.27it/s]


In [4]:
print(f"Number of circuits loaded = {len(loaded_circuits)}")

Number of circuits loaded = 6559


In [5]:
import os
from qiskit import transpile
from qiskit.qpy import load
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeLimaV2
from qiskit_ibm_runtime.fake_provider import FakeFez
from qiskit.result import Result
from qiskit.circuit import QuantumCircuit
from typing import List, Tuple

In [8]:
def simulate_circuits_with_fake_lima_v2(circuits: List[QuantumCircuit], shots: int = 1024) -> List[Tuple[Result, Result]]:
    results = []

    # Define Fake backend
    fake_backend = FakeFez()
    noisy_sim = AerSimulator.from_backend(fake_backend)
    ideal_sim = AerSimulator()

    for idx, qc in enumerate(circuits):
        print(f"\nSimulating circuit {idx + 1}/{len(circuits)}")

        # Transpile for both simulators
        tqc_noisy = transpile(qc, backend=noisy_sim)
        tqc_ideal = transpile(qc, backend=ideal_sim)

        # Run
        job_noisy = noisy_sim.run(tqc_noisy, shots=shots)
        job_ideal = ideal_sim.run(tqc_ideal, shots=shots)

        results.append((job_ideal.result(), job_noisy.result()))

        break

    return results

In [9]:
results = simulate_circuits_with_fake_lima_v2(loaded_circuits)


Simulating circuit 1/6559


In [10]:
results

[(Result(backend_name='aer_simulator', backend_version='0.17.1', job_id='cfd316e1-f316-4c61-96bd-609c28673e52', success=True, results=[ExperimentResult(shots=1024, success=True, meas_level=2, data=ExperimentResultData(counts={'0x25b': 1024}), header={'creg_sizes': [['meas', 10]], 'global_phase': 0.0, 'memory_slots': 10, 'n_qubits': 10, 'name': '10-qubit Deutsch Jozsa malicious', 'qreg_sizes': [['q', 10]], 'metadata': {}}, status=DONE, seed_simulator=64414700, metadata={'time_taken': 0.001303075, 'num_bind_params': 1, 'parallel_state_update': 4, 'parallel_shots': 1, 'required_memory_mb': 1, 'input_qubit_map': [[9, 9], [8, 8], [7, 7], [6, 6], [5, 5], [4, 4], [3, 3], [2, 2], [1, 1], [0, 0]], 'method': 'statevector', 'device': 'CPU', 'num_qubits': 10, 'sample_measure_time': 0.000678992, 'active_input_qubits': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 'num_clbits': 10, 'remapped_qubits': False, 'runtime_parameter_bind': False, 'max_memory_mb': 15772, 'noise': 'ideal', 'measure_sampling': True, 'batch

In [11]:
from collections import Counter

def counts_to_z_expectation(counts: dict, n_qubits: int) -> list:
    """
    Compute expectation values ⟨Z⟩ for each qubit from measurement counts.

    Parameters:
        counts (dict): Dictionary of bitstring outcomes and their frequencies.
        n_qubits (int): Number of qubits (bitstring length)

    Returns:
        List[float]: ⟨Z⟩ expectation values per qubit
    """
    total_shots = sum(counts.values())
    expectations = [0.0] * n_qubits

    for bitstring_hex, count in counts.items():
        # Convert hex to binary with fixed width
        bitstring = bin(int(bitstring_hex, 16))[2:].zfill(n_qubits)
        bitstring = bitstring[::-1]  # Reverse to match Qiskit's qubit ordering

        for i in range(n_qubits):
            z = 1 if bitstring[i] == '0' else -1
            expectations[i] += z * count

    return [round(e / total_shots, 4) for e in expectations]

In [12]:
# Unpack your results
ideal_result, noisy_result = results[0]  # Replace `results` with your actual list

# Extract counts
ideal_counts = ideal_result.results[0].data.counts
noisy_counts = noisy_result.results[0].data.counts

# Compute expectations
n_qubits = 10
ideal_z = counts_to_z_expectation(ideal_counts, n_qubits)
noisy_z = counts_to_z_expectation(noisy_counts, n_qubits)

# Display
print("⟨Z⟩ values (ideal):", ideal_z)
print("⟨Z⟩ values (noisy):", noisy_z)


⟨Z⟩ values (ideal): [-1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
⟨Z⟩ values (noisy): [-0.9531, -0.9922, 0.9961, -0.9492, -0.9609, 0.9961, -0.9648, 0.998, 0.9922, -0.9258]


In [24]:
import os
import json
from pathlib import Path
from typing import List, Dict
from concurrent.futures import TimeoutError as FuturesTimeoutError

from qiskit.qpy import load
from qiskit_aer import AerSimulator
from qiskit import transpile
# from qiskit_ibm_runtime.fake_provider import   # keep your chosen fake backend

def simulate_and_store_z_expectations_json(
    qpy_folder: str,
    output_json_path: str,
    shots: int = 1024,
    per_circuit_timeout_s: float = 30.0,
) -> None:
    """
    Load QPY circuits, simulate (ideal & noisy), compute <Z> per qubit, and
    save to a JSON file. Skips any circuit whose simulation exceeds the timeout.

    Parameters
    ----------
    qpy_folder : str
        Folder path containing .qpy files.
    output_json_path : str
        Path to save the output JSON.
    shots : int
        Number of shots for each simulation.
    per_circuit_timeout_s : float
        Seconds to wait for each simulator job before skipping.
    """

    def counts_to_z_expectation(counts: Dict[str, int], n_qubits: int) -> List[float]:
        """Supports hex ('0x...') and binary ('0101') keys."""
        total_shots = sum(counts.values())
        if total_shots == 0:
            return [0.0] * n_qubits

        expectations = [0.0] * n_qubits
        for key, cnt in counts.items():
            if key.startswith("0x") or key.startswith("0X"):
                val = int(key, 16)
            else:
                # treat as binary bitstring like '0101'
                # if something else, this will raise which is fine
                val = int(key, 2)
            bitstring = bin(val)[2:].zfill(n_qubits)[::-1]  # little-endian
            for i in range(n_qubits):
                z = 1 if bitstring[i] == "0" else -1
                expectations[i] += z * cnt

        return [round(e / total_shots, 6) for e in expectations]

    def unique_key(preferred: str, fallback_base: str, taken: set) -> str:
        base = preferred.strip() if preferred and preferred.strip() else fallback_base
        if base not in taken:
            taken.add(base)
            return base
        k = 2
        while True:
            candidate = f"{base}#{k}"
            if candidate not in taken:
                taken.add(candidate)
                return candidate
            k += 1

    results_dict: Dict[str, Dict[str, List[float]]] = {}

    fake_backend = FakeFez()
    noisy_sim = AerSimulator.from_backend(fake_backend)
    ideal_sim = AerSimulator()

    taken_names = set()

    for file in tqdm(sorted(os.listdir(qpy_folder))):
        if not file.endswith(".qpy"):
            continue

        file_path = os.path.join(qpy_folder, file)
        try:
            with open(file_path, "rb") as f:
                circuits = load(f)  # returns a list of QuantumCircuit objects
        except Exception as e:
            print(f"⚠️ Error loading {file}: {e}")
            continue

        for idx, qc in enumerate(circuits):
            n_qubits = qc.num_qubits
            # circuit_key = unique_key(qc.name, f"{Path(file).stem}_{idx}", taken_names)
            circuit_key = file
            # print(f"• Simulating '{circuit_key}' ({n_qubits} qubits)...")

            try:
                tqc_ideal = transpile(qc, backend=ideal_sim)
                tqc_noisy = transpile(qc, backend=noisy_sim)

                job_ideal = ideal_sim.run(tqc_ideal, shots=shots)
                job_noisy = noisy_sim.run(tqc_noisy, shots=shots)

                ideal_result = job_ideal.result(timeout=per_circuit_timeout_s)
                noisy_result = job_noisy.result(timeout=per_circuit_timeout_s)

                # Use the first experiment's counts (each run has one circuit here)
                ideal_counts = ideal_result.results[0].data.counts
                noisy_counts = noisy_result.results[0].data.counts

                z_ideal = counts_to_z_expectation(ideal_counts, n_qubits)
                z_noisy = counts_to_z_expectation(noisy_counts, n_qubits)

                results_dict[circuit_key] = {
                    "z_ideal": z_ideal,
                    "z_noisy": z_noisy,
                }


            except FuturesTimeoutError:
                print(f"⏱️ Timeout (> {per_circuit_timeout_s:.0f}s). Skipping '{circuit_key}'.")
                continue
            except Exception as e:
                print(f"❌ Error simulating '{circuit_key}': {e}")
                continue
        # break

            

    with open(output_json_path, "w") as json_out:
        json.dump(results_dict, json_out, indent=2)

    print(f"\n✅ Results saved to {output_json_path}")


In [ ]:
simulate_and_store_z_expectations_json(
    qpy_folder="/home/alitousi/projects/quantum/ExecutionResults/StoredCircuits/",
    output_json_path="z_expectations.json",
    shots=1024
)


  0%|▎                                                                                            | 21/7004 [01:41<27:16:29, 14.06s/it]

⏱️ Timeout (> 30s). Skipping '00b58218-c1bf-4c3e-8efb-bae8d8b5a2a4.qpy'.


  1%|▊                                                                                             | 61/7004 [03:50<5:32:10,  2.87s/it]

⚠️ Error loading 02526ca5-0bbb-4a66-99ab-c1fcb4c17062.qpy: MCMTGate.__init__() missing 3 required positional arguments: 'gate', 'num_ctrl_qubits', and 'num_target_qubits'


  1%|▉                                                                                             | 70/7004 [04:14<6:02:27,  3.14s/it]

...

  9%|███████▋                                                                            | 640/7004 [13:47:18<16271:42:51, 9204.62s/it]

⏱️ Timeout (> 30s). Skipping '1781b750-a36e-4f51-b852-556a8de941f5.qpy'.


 10%|████████▉                                                                                 | 691/7004 [13:50:50<9:29:24,  5.41s/it]

⚠️ Error loading 19565926-4546-4618-95cb-0248ba424b52.qpy: MCMTGate.__init__() missing 3 required positional arguments: 'gate', 'num_ctrl_qubits', and 'num_target_qubits'


 10%|█████████                                                                                 | 706/7004 [13:51:35<6:13:46,  3.56s/it]